In [2]:
import torch
from torch import nn, Tensor
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
import numpy as np
from flow_matching.solver import ODESolver
from torch.utils.data import TensorDataset, DataLoader
import time
import torch.nn.functional as F

import sys
sys.path.append('./Textual-Anomaly-Detection-Framework/Anomaly Detection Framework')
import Modelisation.evaluation as ev
from utils import load_data_inlier, load_data_test
from utils import save_results

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Model

In [ ]:
class FlowMatching(nn.Module):
    def __init__(self, source, target, input_dim=64, latent_dim=256, device='cuda', seed=42):
        super().__init__()
        
        self.seed = seed
        self.target = target
        self.source = source
        self.device = device
            
        if self.source == 'target-noised':
            noise = torch.randn(self.target.shape[0], self.target.shape[1])
            self.target_noised = self.target + noise
            
        if self.source == 'point-mean':
            self.point_mean = self.target.mean(dim=0).to(self.device)            
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, latent_dim), nn.ELU(),
            nn.Linear(latent_dim, latent_dim), nn.ELU(),
            nn.Linear(latent_dim, latent_dim), nn.ELU(),
            nn.Linear(latent_dim, input_dim)
        )

    def forward(self, x, t):
        t = t.expand(x.shape[0], 1)            
        xt = torch.cat([x, t], dim=1)

        return self.net(xt)
    
    def sampling_source(self, n_samples):

        
        if self.source == 'gaussian':
            return torch.randn(n_samples, self.input_dim).to(self.device)
    
        if self.source == 'circle': return Tensor(make_circles(n_samples=n_samples, noise=0.05, factor=0.95)[0]).to(self.device)
    
        if self.source == 'poisson': return Tensor(np.random.poisson(5, (n_samples, self.input_dim))).to(self.device)
    
        if self.source == 'uniform': return Tensor(np.random.uniform(low=self.target.min(), high=self.target.max(), size=(n_samples, self.input_dim))).to(self.device)
    
        if self.source == 'sphere':
            
            z = torch.randn(n_samples, self.input_dim)
            return Tensor(z / z.norm(dim=1, keepdim=True)).to(self.device)
        
        if self.source == 'sphere-noised':
            z = torch.randn(n_samples, self.input_dim)
                   
            z = z / z.norm(dim=1, keepdim=True)
            noise = torch.randn_like(z) * 0.25
            return Tensor(z + noise).to(self.device)
        
        if self.source == 'point-mean':
            # return self.point_mean.repeat(n_samples).reshape(-1,self.input_dim)
            return Tensor(np.array([0, 0]).repeat(n_samples).reshape(-1,self.input_dim)).to(self.device)
            
        
    def interpolation(self, n_samples, x1=None):

        # source sampling
        x0 = self.sampling_source(n_samples)
        # target sampling ( get n_samples examples from the all target dataset )
        # replace --> avec ou sans remise
        if x1 is None : idx = np.random.choice(self.target.shape[0], n_samples, replace=True)
        
        if self.source == 'target-noised':
            noise = torch.randn(x1.shape[0], x1.shape[1]).to(self.device)
            x0 = x1 + noise
        
        if x1 is None : x1 = self.target[idx].to(self.device)

        # sampling the time between 0 ad 1
        t = torch.rand(n_samples, 1).to(self.device)
        # t = torch.rand(n_samples).to(self.device)

        xt = (1 - t) * x0 + t * x1
        ut = x1 - x0
        
        return xt, t, ut

## Trainer

In [ ]:
class FlowMatchingTrainer():
    def __init__(self, flow_model,  verbose=True):

        self.flow_model = flow_model
        self.verbose = verbose

    def train(self, dataloader, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam'):
        
        if optimizer_type == 'adam':
            optimizer = torch.optim.Adam(self.flow_model.parameters(), lr=lr, weight_decay=weight_decay)

        for s in range(n_epochs):

            for data, in dataloader:

                xt, t, ut = self.flow_model.interpolation(data.shape[0], data)
                vt =  self.flow_model(xt, t)

                optimizer.zero_grad()

                loss = loss_fn(vt, ut)
                loss.backward()

                optimizer.step()
            
            if self.verbose and s % (n_epochs // 5) == 0:
                print(f" step {s} -> loss : {loss.item():.5f}")
                
        return self.flow_model

    def forward_flow(self, x_0, solver_type='midpoint', n_steps=10):
            
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target = solver.sample(x_init=x_0, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target

    def backward_flow(self,x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_target_to_source
    
    def forward_backward_flow(self, x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target_recons = solver.sample(x_init=x_inter_target_to_source[-1], method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target_recons


    def test(self, X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10):

        if score_type == 'norm':
            x_source_after_backward = self.backward_flow(X_test, solver_type, n_steps)[-1].cpu().detach()
            # scores = ((x_source_after_backward - self.flow_model.target_centroid) ** 2).sum(dim=1)
            scores = (x_source_after_backward ** 2).sum(dim=1)

        if score_type == 'recons':
            x_target_after_forward_backward = self.forward_backward_flow(X_test, solver_type, n_steps)[-1]
            scores = ((torch.norm(x_target_after_forward_backward - X_test, dim=1)** 2)).cpu().detach()
 
        # auc = roc_auc_score(y_test, scores)
        # ap = average_precision_score(y_test, scores)
        # fpr, tpr, thresholds = roc_curve(y_test, scores)
        # idx = np.where(tpr >= 0.95)[0][0]
        # fpr95 = fpr[idx]

        auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)

        print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  

        return auc, fpr95, ap 

## Data

In [ ]:
inlier_topic = 'trade'
dataset_name = 'reuters'
type_tac = 'ruff'
anomaly_rate = 0.1
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
n_run = 2

data_train = load_data_inlier(dataset_name, inlier_topic, save_dir, is_infec=False, is_cvdd=True)
data_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir, is_cvdd=True)

print(data_train)
print(data_test)

X_inlier = Tensor(data_train['sbert_embeddings']).to(device)
X_test =  Tensor(data_test['sbert_embeddings']).to(device)
y_test = np.array(data_test['anomaly_class'])

## Run

In [ ]:
batch_size = 64
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

input_dim = X_inlier.shape[1]
latent_dim = 256
lr = 1e-5
weight_decay = 1e-5
n_epochs = 1000

target = X_inlier.cpu()
source = 'target-noised'

flow_model = FlowMatching(source, target, input_dim, latent_dim, device).to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

fm_trainer = FlowMatchingTrainer(flow_model, verbose=True)

flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='recons', solver_type='midpoint', n_steps=10)


In [ ]:
# x_inter_target_to_source = fm_trainer.backward_flow(X_test)[-1]
# scores = (x_inter_target_to_source ** 2).sum(dim=1)

In [ ]:
# x_inter_target_to_source_inlier = x_inter_target_to_source[:, y_test == 0, :]
# x_inter_target_to_source_inlier.shape
# x_inter_target_to_source_anomaly = x_inter_target_to_source[:, y_test == 1, :]
# x_inter_target_to_source_anomaly.shape

## Toy Example

In [ ]:
# a = np.concatenate([make_moons(300, noise=0.05)[0] , torch.randn(30, 2)])
a = np.concatenate([make_circles(400, noise=0.05, factor=0.15)[0] , torch.randn(0, 2)])
plt.scatter(a[:,0], a[:,1])

In [ ]:
batch_size = 256
# X_train = TensorDataset(Tensor(np.concatenate([make_moons(300, noise=0.05)[0] , torch.randn(30, 2)])).to(device))
# X_train = TensorDataset(Tensor(make_moons(256, noise=0.05)[0]).to(device))
X_train = Tensor(make_circles(noise=0.05, factor=0.15)[0]).to(device)
# X_train = Tensor(np.array([0, 0]).repeat(300).reshape(-1,2)).to(device)
# X_train = Tensor(torch.randn(300, 2)).to(device)
X_inlier_dl = DataLoader(TensorDataset(X_train), batch_size=batch_size, shuffle=True)

input_dim = 2
latent_dim = 64
lr = 1e-2
weight_decay = 0
n_epochs = 10000

target = X_train.cpu()
source = 'gaussian'

flow_model = FlowMatching(source, target, input_dim, latent_dim, device).to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

fm_trainer = FlowMatchingTrainer(flow_model, verbose=True)

flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

In [ ]:
# x_0 = torch.randn(300, 2).to(device)
# x_0 = Tensor(np.concatenate([make_moons(300, noise=0.05)[0] , torch.randn(30, 2)])).to(device)
# x_0 = Tensor(np.concatenate([make_circles(n_samples=300, noise=0.05, factor=0.95)[0] , torch.randn(30, 2)])).to(device)
x_0 = flow_model_trained.sampling_source(300)
# x_0 = Tensor([0]).to(device)
x_inter_source_to_target = fm_trainer.forward_flow(x_0)

n_steps = x_inter_source_to_target.shape[0]

fig, axes = plt.subplots(1, n_steps, figsize=(30, 4), sharex=True, sharey=True)
time_steps = torch.linspace(0, 1.0, n_steps)

for i in range(n_steps):
    axes[i].scatter(
        x_inter_source_to_target[i, :, 0].cpu(),
        x_inter_source_to_target[i, :, 1].cpu(),
        s=10
    )
    axes[i].set_title(f't = {time_steps[i]:.2f}')
    axes[i].set_xlim(-3.0, 3.0)
    axes[i].set_ylim(-3.0, 3.0)

plt.tight_layout()
plt.show()


In [ ]:
# x_1 = Tensor(make_moons(300, noise=0.05)[0]).to(device)
x_1 = Tensor(make_circles(n_samples=300, noise=0.05, factor=0.15)[0]).to(device)
# x_1 = Tensor(np.array([0, 0]).repeat(300).reshape(-1,2)).to(device)
# x_1 = Tensor(torch.randn(300, 2)).to(device)
# x_1 = Tensor(np.concatenate([make_moons(300, noise=0.05)[0] , torch.randn(30, 2)])).to(device)
x_inter_target_to_source = fm_trainer.backward_flow(x_1)

In [ ]:
n_steps = x_inter_target_to_source.shape[0]

fig, axes = plt.subplots(1, n_steps, figsize=(30, 4), sharex=True, sharey=True)
time_steps = torch.linspace(1., 0.0, n_steps)

for i in range(n_steps):
    axes[i].scatter(
        x_inter_target_to_source[i, :, 0].cpu(),
        x_inter_target_to_source[i, :, 1].cpu(),
        s=10
    )
    axes[i].set_title(f't = {time_steps[i]:.2f}')
    axes[i].set_xlim(-3.0, 3.0)
    axes[i].set_ylim(-3.0, 3.0)

plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def get_point_velocity(model, traj_point, device):
    
    velocities = []

    for i in range(traj_point.shape[0]):
        x = traj_point[i].unsqueeze(0).to(device)  
        t = torch.tensor([[i / (traj_point.shape[0] - 1)]], device=device)
        v = model(x, t) 
        velocities.append(v.cpu().squeeze(0))

    return torch.stack(velocities)

In [ ]:
point_idx = 18
traj_point = x_inter_source_to_target[:, point_idx, :]

In [ ]:
t_idx = 1
vel_point = get_point_velocity(flow_model_trained, traj_point, device)

fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(
    x_inter_source_to_target[t_idx, :, 0].cpu(),
    x_inter_source_to_target[t_idx, :, 1].cpu(),
    s=30,
    alpha=0.3
)

x, y = traj_point[t_idx].cpu()
vx, vy = vel_point[t_idx].cpu()

ax.scatter(x, y, c="red", s=20, zorder=3)

ax.quiver(
    x, y,
    vx, vy,
    angles="xy",
    scale_units="xy",
    scale=1.0,
    color="red",
    width=0.006,
    zorder=4
)

ax.set_title(f"t = {t_idx / 9:.2f}")
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect("equal")

plt.show()

In [ ]:
plt.figure(figsize=(5, 5))

plt.scatter(
    x_inter_source_to_target[-1, :, 0].cpu(),
    x_inter_source_to_target[-1, :, 1].cpu(),
    s=10, alpha=0.3
)

plt.plot(
    traj_point[:, 0].cpu(),
    traj_point[:, 1].cpu(),
    "-o",
    c="red",
    label="trajectory"
)

plt.legend()
plt.axis("equal")
plt.show()

## AEFM

Two strategies of training can be done : 
- Train separately the **Flow Encoder** and the **Flow Decoder** and then train them together with a **reconstruction loss**
- Train the **Flow Encoder** and the **Flow Decoder** jointly with a global loss composed of **Flow Encoder FM Loss,** **Flow Decoder FM Loss** and the **reconstruction loss**

### Separated Training

In [ ]:
class FlowMatching(nn.Module):
    def __init__(self, source, target, input_dim=64, latent_dim=256, device='cuda', seed=42):
        super().__init__()
        
        self.seed = seed
        self.target = target
        self.source = source
        self.device = device
                         
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, latent_dim), nn.ELU(),
            nn.Linear(latent_dim, latent_dim), nn.ELU(),
            nn.Linear(latent_dim, latent_dim), nn.ELU(),
            nn.Linear(latent_dim, input_dim)
        )

    def forward(self, x, t):
        t = t.expand(x.shape[0], 1)            
        xt = torch.cat([x, t], dim=1)

        return self.net(xt)
    
    def sampling(self, n_samples):

        return torch.randn(n_samples, self.input_dim).to(self.device)

    def interpolation(self, samples, phase='encoder'):

        n_samples = samples.shape[0]

        if phase == 'encoder':    
            x0 = samples
            x1 = self.sampling(n_samples)
        
        if phase == 'decoder':
            x0 = self.sampling(n_samples)
            x1 = samples

        # sampling the time between 0 ad 1
        t = torch.rand(n_samples, 1).to(self.device)
        # t = torch.rand(n_samples).to(self.device)

        xt = (1 - t) * x0 + t * x1
        ut = x1 - x0
        
        return xt, t, ut
    

class FlowMatchingAETrainer():
    def __init__(self, flow_model, type='encoder',  verbose=True):

        self.flow_model = flow_model
        self.type = type
        self.verbose = verbose

    def train(self, dataloader, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam'):
        
        if optimizer_type == 'adam':
            optimizer = torch.optim.Adam(self.flow_model.parameters(), lr=lr, weight_decay=weight_decay)

        for s in range(n_epochs):

            for data, in dataloader:

                xt, t, ut = self.flow_model.interpolation(data, self.type)
                vt =  self.flow_model(xt, t)

                optimizer.zero_grad()

                loss = loss_fn(vt, ut)
                loss.backward()

                optimizer.step()
            
            if self.verbose and s % (n_epochs // 5) == 0:
                print(f" step {s} -> loss : {loss.item():.5f}")
                
        return self.flow_model

    def forward_flow(self, x_0, solver_type='midpoint', n_steps=10):
            
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target = solver.sample(x_init=x_0, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target

    def backward_flow(self,x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_target_to_source
    
    def backward_forward_flow(self, x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target_recons = solver.sample(x_init=x_inter_target_to_source[-1], method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target_recons
    
    def forward_backward_flow(self, x_0, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target_recons = solver.sample(x_init=x_0, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)
        
        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_inter_source_to_target_recons[-1], method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_target_to_source


    def test(self, X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10):

        if score_type == 'norm':
            x_source_after_backward = self.backward_flow(X_test, solver_type, n_steps)[-1].cpu().detach()
            # scores = ((x_source_after_backward - self.flow_model.target_centroid) ** 2).sum(dim=1)
            scores = (x_source_after_backward ** 2).sum(dim=1)

        if score_type == 'recons':
            x_target_after_forward_backward = self.forward_backward_flow(X_test, solver_type, n_steps)[-1]
            scores = ((torch.norm(x_target_after_forward_backward - X_test, dim=1)** 2)).cpu().detach()
 
        auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)

        print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  

        return auc, fpr95, ap 

In [ ]:
inlier_topic = 'computer'
dataset_name = '20newsgroups'
type_tac = 'ruff'
anomaly_rate = 0.1
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
data_train = load_data_inlier(dataset_name, inlier_topic, save_dir, is_infec=False, is_cvdd=True)
X_inlier = Tensor(data_train['sbert_embeddings']).to(device)

for n_run in range(1,11):
    # n_run = 3

    data_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir, is_cvdd=True)
    X_test =  Tensor(data_test['sbert_embeddings']).to(device)
    y_test = np.array(data_test['anomaly_class'])

    list_auc_fm = []
    list_fpr_fm = []
    list_ap_fm = []

    input_dim = X_inlier.shape[1]
    latent_dim = 256

    ###########################
    ######### ENCODER #########
    ###########################
    source = X_inlier.cpu()
    target = 'gaussian'
    flow_encoder = FlowMatching(source, target, input_dim, latent_dim, device).to(device)

    fmae_encoder_trainer = FlowMatchingAETrainer(flow_encoder, 'encoder')

    batch_size = 64
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)
    lr = 1e-4
    weight_decay = 1e-5
    n_epochs = 50
    loss_fn = nn.MSELoss()

    _ = fmae_encoder_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

    ###########################
    ######### DECODER #########
    ###########################

    source = 'gaussian'
    target = X_inlier.cpu() 
    flow_decoder = FlowMatching(source, target, input_dim, latent_dim, device).to(device)

    fmae_decoder_trainer = FlowMatchingAETrainer(flow_decoder, 'decoder')

    batch_size = 64
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)
    lr = 1e-4
    weight_decay = 1e-5
    n_epochs = 50
    loss_fn = nn.MSELoss()

    _ = fmae_decoder_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')


    ###### INFERENCE ######
    x_0 = X_test
    sbert_to_gaussian = fmae_encoder_trainer.forward_flow(x_0)[-1]
    gaussian_to_sbert = fmae_decoder_trainer.forward_flow(sbert_to_gaussian)[-1]

    scores = ((torch.norm(gaussian_to_sbert - X_test, dim=1)** 2)).cpu().detach()

    auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=True)
    list_ap_fm.append(ap)
    list_auc_fm.append(auc)
    list_fpr_fm.append(fpr95)


In [ ]:
np.mean(list_auc_fm)

### Jointly Training

In [ ]:
class FlowMatching(nn.Module):
    def __init__(self, source, target, input_dim=64, latent_dim=256, device='cuda', seed=42):
        super().__init__()
        
        self.seed = seed
        self.target = target
        self.source = source
        self.device = device
                         
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, latent_dim), nn.Tanh(),
            nn.Linear(latent_dim, latent_dim), nn.Tanh(),
            nn.Linear(latent_dim, latent_dim), nn.Tanh(),
            nn.Linear(latent_dim, input_dim)
        )

    def forward(self, x, t):
        t = t.expand(x.shape[0], 1)            
        xt = torch.cat([x, t], dim=1)

        return self.net(xt)
    
    def sampling(self, n_samples):

        return torch.randn(n_samples, self.input_dim).to(self.device)

    def interpolation(self, samples, phase='encoder', t=None):

        n_samples = samples.shape[0]

        if phase == 'encoder':    
            x0 = samples
            x1 = self.sampling(n_samples)
        
        if phase == 'decoder':
            x0 = self.sampling(n_samples)
            x1 = samples

        # sampling the time between 0 ad 1
        if t is None:
            t = torch.rand(n_samples, 1).to(self.device)
        # t = torch.rand(n_samples).to(self.device)

        xt = (1 - t) * x0 + t * x1
        ut = x1 - x0
        
        return xt, t, ut
    

class FlowMatchingAETrainer():
    def __init__(self, flow_encoder, flow_decoder, lambda_1, lambda_2, lambda_3, device=device, verbose=True):

        self.flow_encoder = flow_encoder
        self.flow_decoder = flow_decoder
        self.lambda_1 = lambda_1
        self.lambda_2 = lambda_2
        self.lambda_3 = lambda_3
        self.device = device
        self.verbose = verbose

    def train(self, dataloader, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam'):
        
        if optimizer_type == 'adam':
            optimizer = torch.optim.Adam(list(self.flow_encoder.parameters()) + list(self.flow_decoder.parameters()), lr=lr, weight_decay=weight_decay)

        for s in range(n_epochs):

            for data, in dataloader:

                t = torch.rand(data.shape[0], 1).to(self.device)

                # Flow Encoder Loss
                xt_encoder, _, ut_encoder = self.flow_encoder.interpolation(data, 'encoder', t)
                vt_encoder =  self.flow_encoder(xt_encoder, t)

                optimizer.zero_grad()
                loss_encoder = loss_fn(vt_encoder, ut_encoder)
                
                # Flow Decoder Loss
                xt_decoder, _, ut_decoder = self.flow_decoder.interpolation(data, 'decoder', t)
                vt_decoder =  self.flow_decoder(xt_decoder, t)

                optimizer.zero_grad()
                loss_decoder = loss_fn(vt_decoder, ut_decoder)

                # Reconstruction Loss
                data_to_gaussian = self.forward_flow(self.flow_encoder, data)[-1]
                data_hat = self.forward_flow(self.flow_decoder, data_to_gaussian)[-1]

                loss_recons = loss_fn(data, data_hat)     

                # Loss           
                loss = self.lambda_1*loss_encoder + self.lambda_2*loss_decoder + self.lambda_3*loss_recons                
                loss.backward()

                optimizer.step()
            
            if self.verbose and s % (n_epochs // 3) == 0:
                print(
                    f"step {s}\n"
                    f"  ├─ loss_encoder     : {loss_encoder.item():.5f}\n"
                    f"  ├─ loss_decoder     : {loss_decoder.item():.5f}\n"
                    f"  ├─ loss_reconstruction     : {loss_recons.item():.5f}\n"
                    f"  └─ loss_total : {loss.item():.5f}"
                )

    def forward_flow(self, flow_model, x_0, solver_type='midpoint', n_steps=10):
            
        solver = ODESolver(velocity_model=flow_model)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target = solver.sample(x_init=x_0, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target

    def backward_flow(self, flow_model, x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_target_to_source
    
    def backward_forward_flow(self, x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target_recons = solver.sample(x_init=x_inter_target_to_source[-1], method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target_recons
    
    def forward_backward_flow(self, x_0, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target_recons = solver.sample(x_init=x_0, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)
        
        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_inter_source_to_target_recons[-1], method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_target_to_source


    def test(self, X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10):

        if score_type == 'norm':
            x_source_after_backward = self.backward_flow(X_test, solver_type, n_steps)[-1].cpu().detach()
            # scores = ((x_source_after_backward - self.flow_model.target_centroid) ** 2).sum(dim=1)
            scores = (x_source_after_backward ** 2).sum(dim=1)

        if score_type == 'recons':
            x_target_after_forward_backward = self.forward_backward_flow(X_test, solver_type, n_steps)[-1]
            scores = ((torch.norm(x_target_after_forward_backward - X_test, dim=1)** 2)).cpu().detach()
 
        auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)

        print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  

        return auc, fpr95, ap 

In [ ]:
inlier_topic = 'Sci-Tech'
dataset_name = 'agnews'
type_tac = 'ruff'
anomaly_rate = 0.1
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
# n_run = 1
data_train = load_data_inlier(dataset_name, inlier_topic, save_dir, is_infec=False, is_cvdd=True)
X_inlier = Tensor(data_train['sbert_embeddings']).to(device)

list_auc_fm = []
list_fpr_fm = []
list_ap_fm = []
list_time_fm = []

for n_run in range(1, 11):

    data_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir, is_cvdd=True)
    X_test =  Tensor(data_test['sbert_embeddings']).to(device)
    y_test = np.array(data_test['anomaly_class'])

    input_dim = X_inlier.shape[1]
    latent_dim = 128

    source = X_inlier.cpu()
    target = 'gaussian'
    flow_encoder = FlowMatching(source, target, input_dim, latent_dim, device).to(device)

    source = 'gaussian'
    target = X_inlier.cpu()
    flow_decoder = FlowMatching(source, target, input_dim, latent_dim, device).to(device)

    lambda_1 = 10.
    lambda_2 = 10.
    lambda_3 = 1.

    fmae_trainer = FlowMatchingAETrainer(flow_encoder, flow_decoder, lambda_1, lambda_2, lambda_3,  verbose=False)

    batch_size = 128
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)
    lr = 1e-4
    weight_decay = 0
    n_epochs = 30
    loss_fn = nn.MSELoss()

    tac = time.time()
    fmae_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs)
    tic = time.time()

    list_time_fm.append((tic-tac))

    data_to_gaussian = fmae_trainer.forward_flow(fmae_trainer.flow_encoder, X_test)[-1]
    X_test_hat = fmae_trainer.forward_flow(fmae_trainer.flow_decoder, data_to_gaussian)[-1]

    scores = ((torch.norm(X_test_hat - X_test, dim=1)** 2)).cpu().detach()
    auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)
    print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}", end="\n\n")  

    list_auc_fm.append(auc)
    list_ap_fm.append(ap)
    list_fpr_fm.append(fpr95)

In [ ]:
save_results(
                dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb='sentence-bert' ,ad_model="flow-matching-AE",
                auc_mean=np.mean(list_auc_fm), ap_mean=np.mean(list_ap_fm),fpr_mean=np.mean(list_fpr_fm),
                auc_std = np.std(list_auc_fm),ap_std =  np.std(list_ap_fm),fpr_std = np.std(list_fpr_fm),
                train_time = np.mean(list_time_fm), nu=0.0, overwrite='naive'
                )

In [ ]:
inlier_topic = 'World'
dataset_name = 'agnews'
type_tac = 'ruff'
anomaly_rate = 0.1
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
n_run = 2
data_train = load_data_inlier(dataset_name, inlier_topic, save_dir, is_infec=False, is_cvdd=True)
X_inlier = Tensor(data_train['sbert_embeddings']).to(device)

data_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir, is_cvdd=True)
X_test =  Tensor(data_test['sbert_embeddings']).to(device)
y_test = np.array(data_test['anomaly_class'])

In [ ]:
input_dim = X_inlier.shape[1]
latent_dim = 128

source = X_inlier.cpu()
target = 'gaussian'
flow_encoder = FlowMatching(source, target, input_dim, latent_dim, device).to(device)

source = 'gaussian'
target = X_inlier.cpu()
flow_decoder = FlowMatching(source, target, input_dim, latent_dim, device).to(device)

lambda_1 = 10.
lambda_2 = 10.
lambda_3 = 1.

fmae_trainer = FlowMatchingAETrainer(flow_encoder, flow_decoder, lambda_1, lambda_2, lambda_3,  verbose=True)

batch_size = 128
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)
lr = 1e-4
weight_decay = 1e-4
n_epochs = 50
loss_fn = nn.MSELoss()

tac = time.time()
fmae_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs)
tic = time.time()

list_time_fm.append((tic-tac))

data_to_gaussian = fmae_trainer.forward_flow(fmae_trainer.flow_encoder, X_test)[-1]
X_test_hat = fmae_trainer.forward_flow(fmae_trainer.flow_decoder, data_to_gaussian)[-1]

scores = ((torch.norm(X_test_hat - X_test, dim=1)** 2)).cpu().detach()
auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)
print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}", end="\n\n")  

## In the neighborhood

In [ ]:
class FlowMatching(nn.Module):
    def __init__(self, source, target, input_dim=64, latent_dim=256, device='cuda', seed=42):
        super().__init__()
        
        self.seed = seed

        self.target = target
        self.source = source
        self.centroid = self.source.mean(dim=0)
        self.cov = 0.01 * np.eye(self.source.shape[1])

        self.device = device
                         
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, latent_dim), nn.Tanh(),
            nn.Linear(latent_dim, latent_dim), nn.Tanh(),
            nn.Linear(latent_dim, latent_dim), nn.Tanh(),
            nn.Linear(latent_dim, input_dim)
        )

    def forward(self, x, t):
        t = t.expand(x.shape[0], 1)            
        xt = torch.cat([x, t], dim=1)

        return self.net(xt)
    
    def sampling(self, n_samples):

        return Tensor(np.random.multivariate_normal(self.centroid, self.cov, n_samples)).to(self.device)

    def interpolation(self, samples):

        n_samples = samples.shape[0]

        x0 = samples
        x1 = self.sampling(n_samples)

        # sampling the time between 0 ad 1
        t = torch.rand(n_samples, 1).to(self.device)
        
        xt = (1 - t) * x0 + t * x1
        ut = x1 - x0
        
        return xt, t, ut

In [ ]:
class FlowMatchingTrainer():
    def __init__(self, flow_model,  verbose=True):

        self.flow_model = flow_model
        self.verbose = verbose

    def train(self, dataloader, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam'):
        
        if optimizer_type == 'adam':
            optimizer = torch.optim.Adam(self.flow_model.parameters(), lr=lr, weight_decay=weight_decay)

        for s in range(n_epochs):

            for data, in dataloader:

                xt, t, ut = self.flow_model.interpolation(data)
                vt =  self.flow_model(xt, t)

                optimizer.zero_grad()

                loss = loss_fn(vt, ut)
                loss.backward()

                optimizer.step()
            
            if self.verbose and s % (n_epochs // 5) == 0:
                print(f" step {s} -> loss : {loss.item():.5f}")
                
        return self.flow_model

    def forward_flow(self, x_0, solver_type='midpoint', n_steps=10):
            
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target = solver.sample(x_init=x_0, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target

    def backward_flow(self,x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_target_to_source
    
    def forward_backward_flow(self, x_1, solver_type='midpoint', n_steps=10):
        solver = ODESolver(velocity_model=self.flow_model)

        time_steps = torch.linspace(1.0, 0.0, n_steps)
        x_inter_target_to_source = solver.sample(x_init=x_1, method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        time_steps = torch.linspace(0.0, 1.0, n_steps)
        x_inter_source_to_target_recons = solver.sample(x_init=x_inter_target_to_source[-1], method=solver_type, step_size=1.0 / n_steps, time_grid=time_steps, return_intermediates=True)

        return x_inter_source_to_target_recons


    def test(self, X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10):

        if score_type == 'norm':
            x_source_after_backward = self.forward_flow(X_test, solver_type, n_steps)[-1].cpu().detach()
            scores = ((x_source_after_backward - self.flow_model.centroid) ** 2).sum(dim=1)
            # scores = (x_source_after_backward ** 2).sum(dim=1)

        if score_type == 'recons':
            x_target_after_forward_backward = self.forward_backward_flow(X_test, solver_type, n_steps)[-1]
            scores = ((torch.norm(x_target_after_forward_backward - X_test, dim=1)** 2)).cpu().detach()
 
        # auc = roc_auc_score(y_test, scores)
        # ap = average_precision_score(y_test, scores)
        # fpr, tpr, thresholds = roc_curve(y_test, scores)
        # idx = np.where(tpr >= 0.95)[0][0]
        # fpr95 = fpr[idx]

        auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)

        print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")  

        return auc, fpr95, ap 

In [ ]:
inlier_topic = 'science'
dataset_name = '20newsgroups'
type_tac = 'ruff'
anomaly_rate = 0.1
save_dir = "/home/2017025/ayouce01/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Data"
# n_run = 6

data_train = load_data_inlier(dataset_name, inlier_topic, save_dir, is_infec=False, is_cvdd=True)
X_inlier = Tensor(data_train['sbert_embeddings']).to(device)

list_auc_fm = []
list_fpr_fm = []
list_ap_fm = []
list_time_fm = []

for n_run in range(1, 11):

    data_test = load_data_test(dataset_name, inlier_topic, n_run, save_dir, is_cvdd=True)
    X_test =  Tensor(data_test['sbert_embeddings']).to(device)
    y_test = np.array(data_test['anomaly_class'])

    batch_size = 64
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    input_dim = X_inlier.shape[1]
    latent_dim = 256
    lr = 1e-3
    weight_decay = 1e-4
    n_epochs = 20

    source = X_inlier.cpu()
    target = 'centroid-neigh'

    flow_model = FlowMatching(source, target, input_dim, latent_dim, device).to(device)
    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    fm_trainer = FlowMatchingTrainer(flow_model, verbose=False)

    tac = time.time()
    flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    tic = time.time()

    auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

    list_auc_fm.append(auc)
    list_ap_fm.append(ap)
    list_fpr_fm.append(fpr95)
    list_time_fm.append((tic-tac))

In [ ]:
save_results(
                dataset_name=dataset_name, inlier_topic=inlier_topic ,type_emb='sentence-bert' ,ad_model="flow-matching-CN",
                auc_mean=np.mean(list_auc_fm), ap_mean=np.mean(list_ap_fm),fpr_mean=np.mean(list_fpr_fm),
                auc_std = np.std(list_auc_fm),ap_std =  np.std(list_ap_fm),fpr_std = np.std(list_fpr_fm),
                train_time = np.mean(list_time_fm), nu=0.0, overwrite='naive'
                )

In [ ]:
mean = np.array([0.0, 0.0])
cov = 0.01 * np.eye(2)

x = np.random.multivariate_normal(mean, cov, 2000)

plt.figure()
plt.scatter(x[:, 0], x[:, 1], alpha=0.4, s=10)
plt.axis("equal")
plt.title("Gaussienne 2D très concentrée")
plt.show()


## Transformers FM

In [ ]:
import math

class SinusoidalPosEmb(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        assert hidden_dim % 2 == 0
        self.hidden_dim = hidden_dim

    def forward(self, x):
        """
        x: Tensor (batch,) ou (batch, 1)
        return: (batch, hidden_dim)
        """
        if x.dim() == 2:
            x = x.squeeze(-1)

        device = x.device
        half_dim = self.hidden_dim // 2

        emb_scale = math.log(10000) / (half_dim - 1)
        emb = torch.exp(
            torch.arange(half_dim, device=device) * -emb_scale
        )

        x = x[:, None] * emb[None, :]
        return torch.cat([torch.sin(x), torch.cos(x)], dim=-1)

In [ ]:
class TextFlowDiT(nn.Module):
    def __init__(self, latent_dim=768, hidden_dim=1024, n_layers=12, n_heads=16):
        super().__init__()

        # Projection initiale
        self.input_proj = nn.Linear(latent_dim, hidden_dim)
        
        # Encoding du timestep
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        
        # Blocs DiT avec adaLN
        self.blocks = nn.ModuleList([
            DiTBlock(hidden_dim, n_heads) for _ in range(n_layers)
        ])
        
        # Tête de prédiction du champ de vecteurs
        self.output_proj = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, z_t, t):
        # z_t: [batch, latent_dim], t: [batch]
        h = self.input_proj(z_t)
        t_emb = self.time_mlp(t)
        
        for block in self.blocks:
            h = block(h, t_emb)  # adaLN conditioning
        
        v = self.output_proj(h)  # Champ de vecteurs
        return v
    

class DiTBlock(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        
        # adaLN: prédit scale et shift à partir du timestep
        self.adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim)  # 6 car scale+shift pour norm1 et norm2 et gate
        )
    
    def forward(self, x, t_emb):
        # x: [batch, dim], t_emb: [batch, dim]
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
            self.adaLN(t_emb).chunk(6, dim=-1)
        
        # Attention avec modulation
        x_norm = modulate(self.norm1(x), shift_msa, scale_msa)
        # Ajoute une dimension sequence (self-attention sur un seul token)
        x_norm = x_norm.unsqueeze(1)  # [batch, 1, dim]
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)  # query, key, value
        attn_out = attn_out.squeeze(1)  # [batch, dim]
        x = x + gate_msa * attn_out
        
        # MLP avec modulation
        x = x + gate_mlp * self.mlp(
            modulate(self.norm2(x), shift_mlp, scale_mlp)
        )
        return x

def modulate(x, shift, scale):
    return x * (1 + scale) + shift

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

# Optionnel: EMA (Exponential Moving Average) pour la stabilité
class EMA:
    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = self.decay * self.shadow[name] + \
                                   (1 - self.decay) * param.data
    
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]
    
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

# Warmup + Cosine decay
def get_lr_schedule(epoch, warmup_epochs, total_epochs, lr):
    if epoch < warmup_epochs:
        return lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return lr * 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159)))

# Training functions
def compute_flow_loss(model, z_0, flow_type='linear', sigma=0.1):
    """
    Calcule la loss pour le flow matching
    
    Args:
        model: le modèle DiT
        z_0: embeddings des textes inliers [batch, latent_dim]
        flow_type: 'linear' ou 'cfm'
        sigma: écart-type pour CFM
    """
    batch_size = z_0.shape[0]
    device = z_0.device
    
    # Sample bruit gaussien
    z_1 = torch.randn_like(z_0)
    
    # Sample timestep uniformément
    t = torch.rand(batch_size, device=device)
    
    if flow_type == 'linear':
        # Flow linéaire: x_t = t*z_1 + (1-t)*z_0
        t_expanded = t.view(-1, 1)
        z_t = t_expanded * z_1 + (1 - t_expanded) * z_0
        
        # Target: vecteur de z_0 vers z_1
        v_target = z_1 - z_0
        
    elif flow_type == 'cfm':
        # Conditional Flow Matching avec variance
        t_expanded = t.view(-1, 1)
        mu_t = t_expanded * z_1 + (1 - t_expanded) * z_0
        
        # Ajoute du bruit gaussien
        eps = torch.randn_like(z_0)
        z_t = mu_t + sigma * eps
        
        # Target reste le même
        v_target = z_1 - z_0
    
    else:
        raise ValueError(f"Unknown flow_type: {flow_type}")
    
    # Prédiction du champ de vecteurs
    v_pred = model(z_t, t)
    
    # Loss MSE
    loss = F.mse_loss(v_pred, v_target)
    
    return loss, v_pred, v_target


def train_epoch(model, dataloader, optimizer, ema, config, epoch):
    """
    Une époque d'entraînement
    """
    model.train()
    total_loss = 0
    total_mse = 0
    
    # pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    
    # for batch_idx, (z_0,) in enumerate(pbar):
    for z_0, in dataloader:
        
        # z_0: [batch, latent_dim] - embeddings des textes
        z_0 = z_0.to(device)
        
        # Forward pass
        loss, v_pred, v_target = compute_flow_loss(
            model, 
            z_0, 
            flow_type=config['flow_type'],
            sigma=config['sigma']
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 
            config['grad_clip']
        )
        
        optimizer.step()
        
        # Update EMA
        ema.update()
        
        # Métriques
        total_loss += loss.item()
        mse = F.mse_loss(v_pred, v_target).item()
        total_mse += mse
        
        # # Update progress bar
        # pbar.set_postfix({
        #     'loss': f"{loss.item():.4f}",
        #     'mse': f"{mse:.4f}",
        #     'lr': f"{optimizer.param_groups[0]['lr']:.6f}"
        # })
    
    avg_loss = total_loss / len(dataloader)
    avg_mse = total_mse / len(dataloader)
    
    return avg_loss, avg_mse


@torch.no_grad()
def validate(model, val_dataloader, config):
    """
    Validation (optionnel si tu as un val set)
    """
    model.eval()
    total_loss = 0
    
    for z_0 in val_dataloader:
        z_0 = z_0.to(device)
        loss, _, _ = compute_flow_loss(
            model, 
            z_0, 
            flow_type=config['flow_type'],
            sigma=config['sigma']
        )
        total_loss += loss.item()
    
    return total_loss / len(val_dataloader)


def save_checkpoint(model, ema, optimizer, epoch, loss, path):
    """
    Sauvegarde un checkpoint
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'ema_shadow': ema.shadow,
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    torch.save(checkpoint, path)
    # print(f"Checkpoint saved to {path}")


def load_checkpoint(model, ema, optimizer, path):
    """
    Charge un checkpoint
    """
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    ema.shadow = checkpoint['ema_shadow']
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    # print(f"Checkpoint loaded from {path}, epoch {epoch}, loss {loss:.4f}")
    return epoch


In [ ]:
batch_size = 16
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

# Configuration
config = {
    'latent_dim': 768,
    'hidden_dim': 512,
    'n_layers': 8,
    'n_heads': 16,
    'lr': 1e-2,
    'weight_decay': 1e-5,
    'epochs': 30,
    'warmup_epochs': 10,
    'grad_clip': 1.0,
    'ema_decay': 0.9999,  # Pour l'EMA du modèle
    'flow_type': 'linear',  # 'linear' ou 'cfm'
    'sigma': 0.1,  # Pour CFM uniquement
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialisation du modèle
model = TextFlowDiT(
    latent_dim=config['latent_dim'],
    hidden_dim=config['hidden_dim'],
    n_layers=config['n_layers'],
    n_heads=config['n_heads']
).to(device)

ema = EMA(model, decay=config['ema_decay'])

# Optimizer et Scheduler
optimizer = AdamW(
    model.parameters(),
    lr=config['lr'],
    weight_decay=config['weight_decay'],
    betas=(0.9, 0.999)
)

In [ ]:
best_loss = float('inf')

print("Starting training...")
print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

for epoch in range(config['epochs']):
    # Adjust learning rate
    lr = get_lr_schedule(epoch, config['warmup_epochs'], config['epochs'], config['lr'])
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    # Train
    train_loss, train_mse = train_epoch(
        model, 
        X_inlier_dl, 
        optimizer, 
        ema, 
        config, 
        epoch
    )
    
    if epoch % (config['epochs'] // 5) == 0:
        print(f"\nEpoch {epoch+1}/{config['epochs']}")
        print(f"Train Loss: {train_loss:.4f}, Train MSE: {train_mse:.4f}, LR: {lr:.6f}")
    
    # Save best model
    if train_loss < best_loss:
        best_loss = train_loss
        # Sauvegarde avec EMA
        ema.apply_shadow()
        save_checkpoint(
            model, ema, optimizer, epoch, train_loss,
            'best_model_ema.pt'
        )
        ema.restore()
        
        # Sauvegarde sans EMA
        save_checkpoint(
            model, ema, optimizer, epoch, train_loss,
            'best_model.pt'
        )
    
    # Checkpoint périodique
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            model, ema, optimizer, epoch, train_loss,
            f'checkpoint_epoch_{epoch+1}.pt'
        )

# print("\nTraining completed!")
# print(f"Best loss: {best_loss:.4f}")

# Sauvegarde finale avec EMA
ema.apply_shadow()
save_checkpoint(
    model, ema, optimizer, config['epochs']-1, train_loss,
    'final_model_ema.pt'
)
ema.restore()


In [ ]:
# Charger le meilleur modèle pour l'inférence
def load_for_inference(model_path='best_model_ema.pt'):
    model = TextFlowDiT(
        latent_dim=config['latent_dim'],
        hidden_dim=config['hidden_dim'],
        n_layers=config['n_layers'],
        n_heads=config['n_heads']
    ).to(device)
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model

# Calculer les scores d'anomalie
model_inference = load_for_inference()

@torch.no_grad()
def compute_anomaly_scores(model, test_embeddings, inlier_embeddings, n_steps=50):
    """
    Score statistique: compare la distribution de z_1 avec celle des inliers
    
    Args:
        inlier_embeddings: échantillon d'embeddings inliers pour estimer la distribution
    """
    model.eval()
    
    # 1. Flow des inliers pour estimer la distribution target empirique
    print("Computing inlier distribution in latent space...")
    z_0_inliers = inlier_embeddings.to(device)
    z_t = z_0_inliers.clone()
    dt = 1.0 / n_steps
    
    for i in range(n_steps):
        t = torch.full((z_0_inliers.shape[0],), i * dt, device=device)
        v = model(z_t, t)
        z_t = z_t + v * dt
    
    z_1_inliers = z_t.cpu().numpy()
    
    # Estimation des statistiques de la distribution inlier dans l'espace latent
    mean_inlier = np.mean(z_1_inliers, axis=0)
    cov_inlier = np.cov(z_1_inliers.T)
    
    # Régularisation pour éviter singularité
    cov_inlier += 1e-6 * np.eye(cov_inlier.shape[0])
    
    # 2. Flow des test samples
    print("Computing test distribution...")
    z_0_test = test_embeddings.to(device)
    z_t = z_0_test.clone()
    
    for i in range(n_steps):
        t = torch.full((z_0_test.shape[0],), i * dt, device=device)
        v = model(z_t, t)
        z_t = z_t + v * dt
    
    z_1_test = z_t.cpu().numpy()
    
    # 3. Score de Mahalanobis par rapport à la distribution inlier
    diff = z_1_test - mean_inlier
    inv_cov = np.linalg.inv(cov_inlier)
    
    # Mahalanobis distance: sqrt((x - μ)^T Σ^(-1) (x - μ))
    scores = np.sqrt(np.sum(diff @ inv_cov * diff, axis=1))
    # z_1_test_tensor = Tensor(np.array((z_1_test ** 2)))
    # scores = z_1_test_tensor.sum(dim=1)
    
    return scores

In [ ]:
# Test
scores = compute_anomaly_scores(model_inference, X_test, X_inlier)
auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)
print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}", end="\n\n")  

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from scipy import stats

# ============= SCORES D'ANOMALIE BASÉS SUR LA DISTRIBUTION TARGET =============

@torch.no_grad()
def compute_anomaly_score_prior_distance(model, test_embeddings, n_steps=50, method='mahalanobis'):
    """
    Score d'anomalie basé sur la distance à la distribution target (prior gaussienne)
    
    Args:
        model: modèle DiT entraîné
        test_embeddings: embeddings de test [batch, latent_dim]
        n_steps: nombre de pas pour résoudre l'ODE
        method: 'euclidean', 'mahalanobis', 'kl', ou 'energy'
    
    Returns:
        scores: scores d'anomalie [batch] (plus haut = plus anormal)
    """
    model.eval()
    z_0 = test_embeddings.to(device)
    batch_size, latent_dim = z_0.shape
    
    # Forward flow: z_0 → z_1 via ODE
    z_t = z_0.clone()
    dt = 1.0 / n_steps
    
    for i in range(n_steps):
        t = torch.full((batch_size,), i * dt, device=device)
        v = model(z_t, t)
        z_t = z_t + v * dt  # Euler step
    
    z_1 = z_t  # Point final dans l'espace latent
    
    # La distribution target est N(0, I)
    # Calculer différentes distances à cette distribution
    
    if method == 'euclidean':
        # Distance euclidienne simple à l'origine
        score = torch.norm(z_1, dim=-1)
    
    elif method == 'mahalanobis':
        # Distance de Mahalanobis (équivalent à euclidienne pour N(0,I))
        # Mais on peut estimer la covariance empirique sur un batch de validation
        score = torch.norm(z_1, dim=-1)
    
    elif method == 'kl':
        # Divergence KL approchée entre N(z_1, I) et N(0, I)
        # KL(N(μ, Σ) || N(0, I)) = 0.5 * (tr(Σ) + μ^T μ - k - log|Σ|)
        # Pour Σ = I: KL = 0.5 * (k + ||μ||^2 - k) = 0.5 * ||μ||^2
        score = 0.5 * torch.sum(z_1 ** 2, dim=-1)
    
    elif method == 'energy':
        # Energy-based score: -log p(z_1) pour p = N(0, I)
        # log p(z) = -0.5 * (k*log(2π) + ||z||^2)
        # Donc -log p(z) = 0.5 * (k*log(2π) + ||z||^2)
        score = 0.5 * torch.sum(z_1 ** 2, dim=-1)
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return score.cpu().numpy()


@torch.no_grad()
def compute_anomaly_score_likelihood(model, test_embeddings, n_steps=100):
    """
    Score basé sur la log-vraisemblance négative avec correction de Jacobien
    
    Plus précis mais plus coûteux en calcul
    """
    model.eval()
    z_0 = test_embeddings.to(device)
    batch_size, latent_dim = z_0.shape
    
    # Forward flow avec accumulation du log-det-jacobien
    z_t = z_0.clone()
    dt = 1.0 / n_steps
    log_det_jacobian = torch.zeros(batch_size, device=device)
    
    for i in range(n_steps):
        t = torch.full((batch_size,), i * dt, device=device)
        
        # Calculer la divergence du champ de vecteurs
        # ∂v/∂z using Hutchinson's trace estimator
        with torch.enable_grad():
            z_t_grad = z_t.clone().requires_grad_(True)
            v = model(z_t_grad, t)
            
            # Trace estimator with random projections
            eps = torch.randn_like(z_t)
            v_eps = torch.sum(v * eps)
            grad_v_eps = torch.autograd.grad(v_eps, z_t_grad, create_graph=False)[0]
            div_v = torch.sum(grad_v_eps * eps, dim=-1)
        
        # Accumulate log-det-jacobian
        log_det_jacobian += div_v * dt
        
        # Euler step
        z_t = z_t + v.detach() * dt
    
    z_1 = z_t
    
    # Log-likelihood sous N(0, I)
    log_prob_prior = -0.5 * torch.sum(z_1 ** 2, dim=-1) - 0.5 * latent_dim * np.log(2 * np.pi)
    
    # Log-likelihood de z_0 = log p(z_1) + log|det J|
    log_prob_z0 = log_prob_prior + log_det_jacobian
    
    # Score d'anomalie = négative log-likelihood
    score = -log_prob_z0
    
    return score.cpu().numpy()


@torch.no_grad()
def compute_anomaly_score_trajectory_deviation(model, test_embeddings, n_steps=50):
    """
    Score basé sur la déviation de la trajectoire par rapport aux trajectoires attendues
    
    Mesure à quel point la trajectoire est "irrégulière"
    """
    model.eval()
    z_0 = test_embeddings.to(device)
    batch_size, latent_dim = z_0.shape
    
    z_t = z_0.clone()
    dt = 1.0 / n_steps
    
    trajectory_deviation = torch.zeros(batch_size, device=device)
    velocity_changes = torch.zeros(batch_size, device=device)
    
    v_prev = None
    
    for i in range(n_steps):
        t = torch.full((batch_size,), i * dt, device=device)
        v = model(z_t, t)
        
        # Mesure 1: Déviation de la magnitude de vitesse attendue
        # Pour un flow linéaire optimal, ||v|| devrait être ~constant = ||z_1 - z_0||
        expected_v_norm = torch.norm(z_0, dim=-1) + 1.0  # Approximation
        actual_v_norm = torch.norm(v, dim=-1)
        trajectory_deviation += torch.abs(actual_v_norm - expected_v_norm)
        
        # Mesure 2: Changement brusque de direction
        if v_prev is not None:
            # Angle entre v_prev et v (via produit scalaire normalisé)
            cos_sim = F.cosine_similarity(v, v_prev, dim=-1)
            velocity_changes += (1 - cos_sim)  # 0 si même direction, 2 si opposé
        
        v_prev = v.clone()
        z_t = z_t + v * dt
    
    z_1 = z_t
    
    # Score final: combinaison de la déviation et de la distance finale
    distance_to_prior = torch.norm(z_1, dim=-1)
    score = trajectory_deviation + velocity_changes + distance_to_prior
    
    return score.cpu().numpy()


@torch.no_grad()
def compute_anomaly_score_statistical(model, test_embeddings, inlier_embeddings, n_steps=50):
    """
    Score statistique: compare la distribution de z_1 avec celle des inliers
    
    Args:
        inlier_embeddings: échantillon d'embeddings inliers pour estimer la distribution
    """
    model.eval()
    
    # 1. Flow des inliers pour estimer la distribution target empirique
    print("Computing inlier distribution in latent space...")
    z_0_inliers = inlier_embeddings.to(device)
    z_t = z_0_inliers.clone()
    dt = 1.0 / n_steps
    
    for i in range(n_steps):
        t = torch.full((z_0_inliers.shape[0],), i * dt, device=device)
        v = model(z_t, t)
        z_t = z_t + v * dt
    
    z_1_inliers = z_t.cpu().numpy()
    
    # Estimation des statistiques de la distribution inlier dans l'espace latent
    mean_inlier = np.mean(z_1_inliers, axis=0)
    cov_inlier = np.cov(z_1_inliers.T)
    
    # Régularisation pour éviter singularité
    cov_inlier += 1e-6 * np.eye(cov_inlier.shape[0])
    
    # 2. Flow des test samples
    print("Computing test distribution...")
    z_0_test = test_embeddings.to(device)
    z_t = z_0_test.clone()
    
    for i in range(n_steps):
        t = torch.full((z_0_test.shape[0],), i * dt, device=device)
        v = model(z_t, t)
        z_t = z_t + v * dt
    
    z_1_test = z_t.cpu().numpy()
    
    # 3. Score de Mahalanobis par rapport à la distribution inlier
    diff = z_1_test - mean_inlier
    inv_cov = np.linalg.inv(cov_inlier)
    
    # Mahalanobis distance: sqrt((x - μ)^T Σ^(-1) (x - μ))
    scores = np.sqrt(np.sum(diff @ inv_cov * diff, axis=1))
    
    return scores


# ============= FONCTION UNIFIÉE AVEC PLUSIEURS MÉTHODES =============

@torch.no_grad()
def compute_anomaly_scores(
    model, 
    test_embeddings, 
    inlier_embeddings=None,
    n_steps=50, 
    methods=['energy', 'trajectory'],
    weights=None
):
    """
    Calcule un score d'anomalie combiné avec plusieurs méthodes
    
    Args:
        model: modèle DiT
        test_embeddings: embeddings de test
        inlier_embeddings: embeddings inliers (pour méthode statistique)
        n_steps: nombre de pas ODE
        methods: liste de méthodes parmi ['euclidean', 'energy', 'trajectory', 'statistical']
        weights: poids pour chaque méthode (si None, moyenne simple)
    
    Returns:
        scores: scores d'anomalie normalisés
    """
    all_scores = []
    
    for method in methods:
        if method == 'euclidean':
            scores = compute_anomaly_score_prior_distance(
                model, test_embeddings, n_steps, method='euclidean'
            )
        elif method == 'energy':
            scores = compute_anomaly_score_prior_distance(
                model, test_embeddings, n_steps, method='energy'
            )
        elif method == 'likelihood':
            scores = compute_anomaly_score_likelihood(
                model, test_embeddings, n_steps
            )
        elif method == 'trajectory':
            scores = compute_anomaly_score_trajectory_deviation(
                model, test_embeddings, n_steps
            )
        elif method == 'statistical':
            if inlier_embeddings is None:
                raise ValueError("inlier_embeddings required for statistical method")
            scores = compute_anomaly_score_statistical(
                model, test_embeddings, inlier_embeddings, n_steps
            )
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Normalisation Z-score
        scores = (scores - np.mean(scores)) / (np.std(scores) + 1e-8)
        all_scores.append(scores)
    
    # Combinaison des scores
    all_scores = np.array(all_scores)
    
    if weights is None:
        weights = np.ones(len(methods)) / len(methods)
    else:
        weights = np.array(weights)
        weights = weights / np.sum(weights)
    
    combined_scores = np.sum(all_scores * weights[:, np.newaxis], axis=0)
    
    return combined_scores


# ============= EXEMPLE D'UTILISATION =============

def evaluate_anomaly_detection(model, test_embeddings, test_labels, inlier_embeddings):
    """
    Évalue les performances de détection d'anomalies
    
    Args:
        test_labels: 0 pour inlier, 1 pour anomalie
    """
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
    import matplotlib.pyplot as plt
    
    # Calcul des scores avec différentes méthodes
    methods_to_test = {
        'Energy': (['energy'], None),
        'Trajectory': (['trajectory'], None),
        'Combined': (['energy', 'trajectory'], [0.6, 0.4]),
        'Statistical': (['statistical'], None),
    }
    
    results = {}
    
    for name, (methods, weights) in methods_to_test.items():
        print(f"\nTesting {name}...")
        
        try:
            scores = compute_anomaly_scores(
                model, 
                test_embeddings,
                inlier_embeddings=inlier_embeddings,
                n_steps=50,
                methods=methods,
                weights=weights
            )
            
            # Métriques
            auroc = roc_auc_score(test_labels, scores)
            auprc = average_precision_score(test_labels, scores)
            auc, fpr95, ap = ev.evaluation(test_labels, scores, verbose=False)
            print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}", end="\n\n")  
            
            results[name] = {
                'scores': scores,
                'auroc': auroc,
                'auprc': auprc
            }
            
            print(f"{name} - AUROC: {auroc:.4f}, AUPRC: {auprc:.4f}")
        
        except Exception as e:
            print(f"Error with {name}: {e}")
    
    # Plot ROC curves
    plt.figure(figsize=(10, 6))
    for name, res in results.items():
        fpr, tpr, _ = roc_curve(test_labels, res['scores'])
        plt.plot(fpr, tpr, label=f"{name} (AUC={res['auroc']:.3f})")
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - Anomaly Detection')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    return results

In [ ]:
model_inference = load_for_inference('best_model_ema.pt')

# Supposons que tu as:
# - test_embeddings: tous tes samples de test
# - test_labels: 0/1 pour inlier/anomalie
# - inlier_embeddings_sample: un échantillon d'inliers pour la méthode statistique

# Exemple simple avec méthode energy
scores = compute_anomaly_scores(
    model_inference,
    X_test,
    n_steps=50,
    methods=['energy']
)
# auc, fpr95, ap = ev.evaluation(y_test, scores, verbose=False)
# print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}", end="\n\n")  

# Ou évaluation complète
results = evaluate_anomaly_detection(
    model_inference,
    X_test,
    y_test,
    X_inlier
)